# Day 021 Project Solution — AI File Organizer

End-to-end: scan a directory → AI-tag each file → write manifest.json and report.csv → organize into category subfolders.

In [ ]:
import json
import csv
import shutil
import tempfile
from pathlib import Path
import ollama


def scan_directory(directory: str, pattern: str = "*") -> list[dict]:
    result = []
    for item in Path(directory).glob(pattern):
        if item.is_file():
            result.append({
                "name": item.name,
                "path": str(item),
                "size_bytes": item.stat().st_size,
                "extension": item.suffix,
            })
    return result


def read_csv(path: str) -> list[dict]:
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: str, rows: list[dict], fieldnames: list[str]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def load_json_files(directory: str) -> list[dict]:
    results = []
    for p in Path(directory).glob("*.json"):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            if isinstance(data, dict):
                data["_source"] = p.name
                results.append(data)
        except Exception:
            pass
    return results


def batch_process_files(directory: str, process_fn) -> list[dict]:
    results = []
    for p in sorted(Path(directory).glob("*")):
        if not p.is_file():
            continue
        try:
            content = p.read_text(encoding="utf-8")
            result = process_fn(content)
            results.append({"path": str(p), "status": "ok", "result": result})
        except Exception as e:
            results.append({"path": str(p), "status": "error", "error": str(e)})
    return results


def ai_tag_file(content: str, model: str = "llama3.2") -> dict:
    system = (
        "You are a file categorization assistant. "
        "Given text content, return JSON with exactly these keys: "
        "category (one word: technical, personal, financial, creative, or other), "
        "tags (list of up to 5 keyword strings), "
        "summary (one sentence describing the content). "
        "Return only valid JSON."
    )
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": f"Categorize this content:\n\n{content[:2000]}"},
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        data = json.loads(raw)
    except Exception:
        data = {}
    return {
        "category": str(data.get("category", "other")).lower().strip() or "other",
        "tags": list(data.get("tags", [])),
        "summary": str(data.get("summary", "")),
    }

## Create Sample Corpus

In [ ]:
CORPUS = {
    'python_tutorial.txt': "Python is a high-level, general-purpose programming language. Created by Guido van Rossum, it was first released in 1991. Python's clean syntax and extensive standard library make it ideal for data science, automation, web development, and AI engineering.",
    'budget_q1.txt': 'Q1 Financial Report: Total revenue $48,500. Operating expenses: rent $12,000, payroll $22,000, cloud services $3,200, marketing $4,100. Net operating income: $7,200. Cash reserves: $31,000.',
    'haiku_autumn.txt': 'Crimson leaves cascade / silence holds the mountain lake / one last heron flies. Frost on morning grass / the old pine bends but does not break / winter teaches patience.',
}

WORK_DIR = Path(tempfile.mkdtemp())
for fname, content in CORPUS.items():
    (WORK_DIR / fname).write_text(content, encoding='utf-8')
print(f'Created {len(CORPUS)} sample files.')

## Step 1 — Scan

In [ ]:
file_list = scan_directory(str(WORK_DIR))
print(f'Scanned {len(file_list)} files:')
for f in file_list:
    print(f"  {f['name']} ({f['size_bytes']} bytes)")

## Step 2 — AI-Tag (scripted: one file per topic)

In [ ]:
results = []
for file_info in file_list:
    content = Path(file_info['path']).read_text(encoding='utf-8')
    tags = ai_tag_file(content)
    row = {**file_info, **tags}
    results.append(row)
    print(f"  {file_info['name']} -> [{tags['category']}] {tags['summary'][:55]}...")

## Step 3 — Write manifest.json

In [ ]:
manifest_path = WORK_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(results, indent=2), encoding='utf-8')
loaded_back = json.loads(manifest_path.read_text(encoding='utf-8'))
print(f'manifest.json: {len(loaded_back)} entries')

## Step 4 — Write and verify report.csv

In [ ]:
report_path = WORK_DIR / 'report.csv'
write_csv(
    str(report_path),
    results,
    fieldnames=['name', 'category', 'summary', 'size_bytes'],
)
report_rows = read_csv(str(report_path))
print(f'report.csv: {len(report_rows)} rows')
for row in report_rows:
    print(f"  {row['name']} | {row['category']}")

## Step 5 — Organize into category folders

In [ ]:
organized = WORK_DIR / 'organized'
organized.mkdir()
for item in results:
    cat_dir = organized / item['category']
    cat_dir.mkdir(exist_ok=True)
    shutil.copy(item['path'], cat_dir / item['name'])
categories = sorted(d.name for d in organized.iterdir() if d.is_dir())
print(f'Organized into {len(categories)} categories: {categories}')

## Summary + Cleanup

In [ ]:
print('\n=== AI File Organizer Summary ===')
print(f'Files processed : {len(results)}')
print(f'Categories found: {sorted(set(r["category"] for r in results))}')
print(f'manifest.json   : {manifest_path.name}')
print(f'report.csv      : {report_path.name}')
shutil.rmtree(WORK_DIR)
print('Temp files cleaned up.')
print('Demo complete!')